# V3 — Alometria e alturas estimadas de árvores de Caxias do Sul (Meta/WRI)

**Notebook independente dos notebooks original e V2. Fonte externa única de altura: Meta/WRI CHMv2 (aproximadamente 1 m).** Usamos os **257 pontos originais de árvores** de `Arvores.gpkg` e os dados de altura pública **já extraídos e guardados no repositório**.

Este estudo confronta o índice empírico C3 (e as variantes C1–C6) com a altura **modelada do dossel** nos pontos. O mapa externo não mede diretamente o indivíduo; o índice C3 não é altura validada em metros.

Fontes: [PDAU Caxias do Sul](https://caxias.rs.gov.br/servicos/meio-ambiente/plano-diretor-de-arborizacao-urbana-pdau), [Meta/WRI CHMv2 — catálogo AWS](https://registry.opendata.aws/dataforgood-fb-forestsv2/) e [Brandt et al. (2026)](https://doi.org/10.1038/s41597-026-07882-0).


## 1. Preparação

Usamos exclusivamente os arquivos já presentes na **Area1** para calcular índices de porte. As alturas Meta/WRI são consultadas no **CSV público reprodutível já salvo no próprio repositório**, sem solicitar novos dados e sem modificar a Area1.


In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Fora do Google Colab: informe o caminho local.")
AREA1=Path("/content/drive/MyDrive/Area1")
SAIDA=Path("/content/v3_saida_alometria")
REPO=Path("/content/mestrado-arvores-urbanas")
if not REPO.exists():
    subprocess.run(["git","clone","--depth","1",
      "https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git",str(REPO)],check=True)
SCRIPT=REPO/"experimentos"/"v3_alometria_caxias.py"
for module in ["numpy","pandas","rasterio","geopandas","skimage","matplotlib"]:
    if importlib.util.find_spec(module) is None:
        pipname="scikit-image" if module=="skimage" else module
        subprocess.run([sys.executable,"-m","pip","install","-q",pipname],check=True)
assert (AREA1/"AOI_RGB_NEW.tif").is_file()
assert (AREA1/"Arvores.gpkg").is_file()


## 2. Verificar o que está disponível

Os pontos existentes em `Arvores.gpkg` identificam árvores, **mas não a espécie ou a altura medida de cada uma**. O PDAU é usado apenas como contexto.


In [ ]:
import geopandas as gpd
g=gpd.read_file(AREA1/"Arvores.gpkg")
print("Pontos:",len(g),"CRS:",g.crs,"Campos:",list(g.columns))


## 3. Cálculo de índices alométricos com os segmentos locais

O Quickshift gera **superpixels, não copas de árvores delimitadas**. O script independente calcula áreas dos segmentos e índices C1–C6. A interpretação de **C3** como altura real em metros não foi validada.


In [ ]:
SAIDA.mkdir(exist_ok=True,parents=True)
subprocess.run([sys.executable,str(SCRIPT),"--area",str(AREA1),
     "--out",str(SAIDA),"--tile","256"],check=True)


## 4. Consultar os resultados e as faixas regionais

As faixas da prefeitura são **por espécie**, e não podem ser atribuídas aos 257 pontos sem identificação botânica.


In [ ]:
import pandas as pd,json
from IPython.display import display,Image
r=json.loads((SAIDA/"v3_resumo.json").read_text(encoding="utf-8"))
display(pd.DataFrame([r]))
display(pd.read_csv(SAIDA/"v3_referencias_alturas_caxias.csv"))
for f in ["v3_histograma_indice_C3.png","v3_sensibilidade_C1_C6.png",
          "v3_faixas_especies_caxias.png"]:
    display(Image(filename=str(SAIDA/f)))


## 5. Sensibilidade adicional (opcional)

Alterar a janela para 512 px pode alterar os superpixels e o índice. Esse teste **não valida a área real da copa**.


In [ ]:
EXECUTAR_512=False
if EXECUTAR_512:
    subprocess.run([sys.executable,str(SCRIPT),"--area",str(AREA1),
        "--out",str(SAIDA.parent/"v3_saida_512"),"--tile","512"],check=True)


## 6. Alturas estimadas Meta/WRI CHMv2 nos 257 pontos da Area1

A amostragem pública Meta/WRI CHMv2 já foi realizada para **todos os 257 pontos**, utilizando o raster original do quadrante `2103121330.tif`. Para cada ponto, mantemos:
- **`altura_meta_pixel_m`**: previsão no pixel da coordenada;
- **`altura_meta_max_5m_m`**: maior previsão no raio de 5 m (teste de sensibilidade; pode vir de outra árvore).

O mapa fornece **previsões modeladas de altura do dossel**, não a altura medida em campo de cada árvore. Nenhuma fonte adicional de altura será carregada.

[Dados da Meta/WRI](https://dataforgood-fb-data.s3.us-east-1.amazonaws.com/forests/v2/global/dinov3_global_chm_v2_ml3/chm/2103121330.tif) · [CSV com 257 pontos](https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas/blob/main/docs/v3_alturas_publicas/v3_comparacao_c3_meta_257.csv) · [Script de extração pública](https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas/blob/main/experimentos/v3_amostrar_altura_publica.py)

Para **atualizar as alturas diretamente na fonte pública**, habilite a variável opcional na próxima célula. Por padrão, o notebook usa o CSV já verificado no repositório, sem novas chamadas externas.


In [ ]:
META_CSV = REPO / "docs" / "v3_alturas_publicas" / "v3_comparacao_c3_meta_257.csv"
REEXTRAIR_ALTURAS_META = False  # Mantenha False para usar os resultados ja coletados.
if REEXTRAIR_ALTURAS_META:
    coordenadas = REPO / "dados_v3" / "pontos_area1_257.csv"
    script_meta = REPO / "experimentos" / "v3_amostrar_altura_publica.py"
    nova_saida = SAIDA / "consulta_meta_publica"
    subprocess.run([sys.executable, str(script_meta),
                    "--pontos", str(coordenadas),
                    "--out", str(nova_saida), "--raio", "5"], check=True)
    META_CSV = nova_saida / "v3_meta_chmv2_estimativas_257_arvores.csv"
assert META_CSV.is_file(), f"Arquivo publico Meta/WRI nao encontrado: {META_CSV}"
print("Fonte unica das alturas de dossel:", META_CSV)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
meta = pd.read_csv(META_CSV, encoding="utf-8-sig")
indices = pd.read_csv(SAIDA / "v3_indices_superpixels_pontos.csv", encoding="utf-8-sig")
campos = ["ponto_id", "altura_meta_pixel_m", "altura_meta_max_5m_m"]
if "altura_meta_pixel_m" not in meta and "altura_meta_v2_pixel_m" in meta:
    meta = meta.rename(columns={
        "altura_meta_v2_pixel_m": "altura_meta_pixel_m",
        "altura_meta_v2_max_raio_5m_m": "altura_meta_max_5m_m"})
if len(meta)!=257 or meta.ponto_id.nunique()!=257 or len(indices)!=257 or indices.ponto_id.nunique()!=257:
    raise ValueError("Esperados exatamente 257 pontos originais e IDs unicos em ambas as tabelas.")
if not set(campos).issubset(meta.columns) or "indice_potencial_C3" not in indices.columns:
    raise ValueError("Colunas de Meta/WRI ou da alometria C3 ausentes.")
dados = indices.merge(meta[campos], on="ponto_id", how="left", validate="one_to_one")
if dados.altura_meta_pixel_m.notna().sum()!=257:
    print("ATENCAO: nem todas as alturas Meta estao disponiveis; consultar cobertura.")
dados["altura_meta_pixel_m"] = pd.to_numeric(dados["altura_meta_pixel_m"], errors="coerce")
dados["altura_meta_max_5m_m"] = pd.to_numeric(dados["altura_meta_max_5m_m"], errors="coerce")
dados.to_csv(SAIDA/"v3_meta_chmv2_c3_257_pontos.csv", index=False, encoding="utf-8-sig")
resumo = {
    "fonte_altura": "Meta/WRI CHMv2; previsao de dossel, nao altura medida",
    "pontos_originais": int(len(dados)),
    "meta_pixel_com_valor": int(dados.altura_meta_pixel_m.notna().sum()),
    "meta_mediana_pixel_m": float(dados.altura_meta_pixel_m.median()),
    "meta_mediana_maximo_raio_5m_m": float(dados.altura_meta_max_5m_m.median()),
    "meta_zeros_pixel": int((dados.altura_meta_pixel_m==0).sum()),
    "meta_zeros_maximo_5m": int((dados.altura_meta_max_5m_m==0).sum()),
    "indice_C3_mediana_sem_unidade_m": float(dados.indice_potencial_C3.median()),
    "pearson_C3_meta_pixel": float(dados[["indice_potencial_C3","altura_meta_pixel_m"]].corr().iloc[0,1]),
    "pearson_C3_meta_maximo_5m": float(dados[["indice_potencial_C3","altura_meta_max_5m_m"]].corr().iloc[0,1]),
    "interpretacao": "Comparacao exploratoria de indices com alturas previstas por modelo externo, sem validacao de campo."
}
(SAIDA/"v3_meta_chmv2_c3_resumo.json").write_text(json.dumps(resumo, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame([resumo]))
fig, ax=plt.subplots(figsize=(7,5))
ax.scatter(dados.indice_potencial_C3, dados.altura_meta_pixel_m, alpha=.55, s=22, label="Pixel no ponto")
ax.scatter(dados.indice_potencial_C3, dados.altura_meta_max_5m_m, alpha=.35, s=22, label="Maximo no raio de 5 m")
ax.set(xlabel="Indice C3 (nao e altura em metros)",ylabel="Altura modelada Meta/WRI CHMv2 (m)",
       title="V3 — Comparacao exploratoria em 257 pontos da Area1")
ax.legend()
fig.tight_layout()
fig.savefig(SAIDA/"v3_meta_chmv2_c3_dispersao.png", dpi=160)
plt.show()


## 7. Conclusões e limites científicos

- **Podemos:** descrever a distribuição e a sensibilidade de C1–C6, comparar os índices com o **mapa Meta/WRI CHMv2** e cotejar com faixas de porte descritas no PDAU.
- **Não podemos:** afirmar a altura real de cada indivíduo, identificar a espécie por esses dados ou calcular **erro contra altura de campo** (MAE/RMSE reais) sem medições locais dos mesmos indivíduos.
- Valor zero no pixel Meta/WRI **não prova inexistência da árvore**. O máximo na vizinhança de 5 m pode incluir outra copa.
- Distâncias a postes não demonstram risco elétrico real.

**Fonte externa de altura adotada na V3: somente Meta/WRI CHMv2.** As etapas de segmentação e os arquivos originais permanecem preservados.
